# Regularization Paths and Model Comparison

**Project question:** How do coefficient paths, cross-validation, and final test performance tell different parts of the shrinkage story?

By the end of this notebook, you should be able to:

- visualize ridge and LASSO coefficient paths on training data
- compare cross-validated RMSE across penalty strengths
- evaluate tuned OLS, ridge, and LASSO workflows on one test set

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

In [ ]:
df = pd.read_csv(DATA / 'simulated_correlated_predictors.csv')
X = df.drop(columns=['id', 'weekly_sales'])
y = df['weekly_sales']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=4031, test_size=0.30
)
alphas = np.logspace(-3, 2, 20)
cv = KFold(n_splits=5, shuffle=True, random_state=4031)

Coefficient paths are descriptive. We fit the scaler and paths on training data only so the final test set remains outside every modeling choice.

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
path_records = []
for alpha in alphas:
    for model_name, estimator in [
        ('ridge', Ridge(alpha=alpha)),
        ('lasso', Lasso(alpha=alpha, max_iter=50000)),
    ]:
        estimator.fit(X_train_scaled, y_train)
        for feature, coefficient in zip(X.columns, estimator.coef_):
            path_records.append({
                'model': model_name, 'alpha': alpha,
                'feature': feature, 'coefficient': coefficient,
            })
paths = pd.DataFrame(path_records)
signal_features = [c for c in X.columns if not c.startswith('noise_')]
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
for ax, model_name in zip(axes, ['ridge', 'lasso']):
    part = paths.query('model == @model_name')
    for feature in signal_features:
        line = part.loc[part['feature'].eq(feature)]
        ax.plot(line['alpha'], line['coefficient'], label=feature)
    ax.set_xscale('log')
    ax.set_title(f'{model_name.title()} coefficient paths')
    ax.set_xlabel('alpha')
axes[0].set_ylabel('coefficient on standardized predictor')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()

Ridge paths shrink continuously toward zero. LASSO paths can reach exact zero. Path shape describes the fitted training sample and should not be interpreted as a sequence of hypothesis tests.

In [ ]:
cv_rows = []
for alpha in alphas:
    for model_name, estimator in [
        ('ridge', Ridge(alpha=alpha)),
        ('lasso', Lasso(alpha=alpha, max_iter=50000)),
    ]:
        workflow = make_pipeline(StandardScaler(), estimator)
        fold_rmse = -cross_val_score(
            workflow, X_train, y_train, scoring='neg_root_mean_squared_error', cv=cv
        )
        cv_rows.append({
            'model': model_name, 'alpha': alpha,
            'cv_rmse_mean': fold_rmse.mean(),
            'cv_rmse_se': fold_rmse.std(ddof=1) / np.sqrt(len(fold_rmse)),
        })
cv_results = pd.DataFrame(cv_rows)
for model_name, part in cv_results.groupby('model'):
    plt.plot(part['alpha'], part['cv_rmse_mean'], marker='o', label=model_name)
plt.xscale('log')
plt.xlabel('alpha')
plt.ylabel('training 5-fold CV RMSE')
plt.legend()
plt.title('Penalty selection uses training data only')

In [ ]:
best = (
    cv_results.sort_values('cv_rmse_mean')
    .groupby('model', as_index=False)
    .first()
    .set_index('model')
)
models = {
    'OLS': LinearRegression(),
    'tuned ridge': make_pipeline(StandardScaler(), Ridge(alpha=best.loc['ridge', 'alpha'])),
    'tuned LASSO': make_pipeline(StandardScaler(), Lasso(alpha=best.loc['lasso', 'alpha'], max_iter=50000)),
}

def rmse(actual, predicted):
    return float(np.sqrt(mean_squared_error(actual, predicted)))

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    rows.append({'model': name, 'test_rmse': rmse(y_test, model.predict(X_test))})
pd.DataFrame(rows).sort_values('test_rmse')

**Interpretation:** Predictive differences should be judged in response units and against their uncertainty, not only by rank. Ridge favors stability; LASSO favors sparsity; OLS remains a useful baseline. None converts an observational association into a causal effect.

**Transfer exercise:** For your project, identify which predictors require encoding or scaling, define a training-only alpha search, and state whether your priority is prediction, sparse communication, or coefficient explanation.